In [ ]:
!pip install opencv-python transformers torch pillow

In [5]:
import cv2
import torch
from PIL import Image
from transformers import BlipForConditionalGeneration, BlipProcessor

MODEL_NAME = "Salesforce/blip-image-captioning-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

processor = BlipProcessor.from_pretrained(MODEL_NAME)
model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)


def caption_image(image: Image.Image) -> str:
    inputs = processor(images=image, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=30)

    return processor.decode(output_ids[0], skip_special_tokens=True)

def describe_video(video_path: str, sample_every_seconds: int = 5) -> dict:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        cap.release()
        raise ValueError("Could not read video FPS")

    frame_interval = max(1, int(fps * sample_every_seconds))

    frame_count = 0
    captions = []

    while cap.isOpened():
        ret, frame = cap.read()

        if not ret:
            break

        if frame_count % frame_interval == 0:
            timestamp = frame_count / fps

            # OpenCV uses BGR, PIL expects RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = Image.fromarray(frame_rgb)

            result = caption_image(image)

            captions.append({
                "timestamp_seconds": round(timestamp, 2),
                "description": result
            })

        frame_count += 1

    cap.release()

    return {
        "video_path": video_path,
        "sample_every_seconds": sample_every_seconds,
        "frame_descriptions": captions
    }

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

In [6]:
describe_video("/home/joel/Documents/music_lesson_app/backend-fastapi/video/waltz_no2_practice.mp4", 1)

{'video_path': '/home/joel/Documents/music_lesson_app/backend-fastapi/video/waltz_no2_practice.mp4',
 'sample_every_seconds': 1,
 'frame_descriptions': [{'timestamp_seconds': 0.0,
   'description': 'a man sitting on a chair holding a cello'},
  {'timestamp_seconds': 1.0,
   'description': 'a man playing a cello in a living room'},
  {'timestamp_seconds': 2.0, 'description': 'a man playing a cello'},
  {'timestamp_seconds': 3.0,
   'description': 'a man sitting on a chair playing a cello'},
  {'timestamp_seconds': 4.0,
   'description': 'a man playing a cello in a living room'}]}